In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%time
!pip install -q pandas numpy scikit-learn xgboost lightgbm \
    rapidfuzz jellyfish \
    sentence-transformers faiss-cpu \
    transformers accelerate datasets \
    torch tqdm unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.2 MB/s eta 0:00:00
CPU times: user 1.67 s, sys: 667 ms, total: 2.34 s
Wall time: 10.8 s


In [ ]:
# Enable 10GB extra virtual RAM on Colab
!fallocate -l 10G /swapfile
!chmod 600 /swapfile
!mkswap /swapfile
!swapon /swapfile
!free -h


Setting up swapspace version 1, size = 10 GiB (10737414144 bytes)
no label, UUID=f905fd4a-751f-4ae4-af1b-19ef43815141
swapon: /swapfile: swapon failed: Invalid argument
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.0Gi       8.3Gi       2.8Mi       3.7Gi        11Gi
Swap:             0B          0B          0B


In [ ]:
import os, re, gc, json, warnings, time
from collections import defaultdict
from pathlib import Path

import pandas as pd
import numpy as np

from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler
import jellyfish
from unidecode import unidecode

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupShuffleSplit
import scipy.sparse as sp

import xgboost as xgb
import lightgbm as lgb

from tqdm.auto import tqdm
tqdm.pandas()

import torch

# Device and VRAM check (fixed 'total_mem' -> 'total_memory')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)

# ============================================================
# PATHS — pointing to your mounted Google Drive directory
# ============================================================
BASE_DIR = Path('/content/drive/MyDrive/amazon/student_resource')
TRAIN_DIR = BASE_DIR / 'dataset' / 'train'
TEST_DIR  = BASE_DIR / 'dataset' / 'test'

# Local output directory on Colab runtime for fast read/write
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Verify path connection
print(f"Drive path exists: {BASE_DIR.exists()}")
if BASE_DIR.exists():
    print("Contents:", [p.name for p in BASE_DIR.iterdir()])

# ============================================================
# HYPERPARAMETERS — tune these
# ============================================================
TFIDF_TOP_K        = 25      # candidates per S1 entity from TF-IDF
SBERT_TOP_K        = 25      # candidates per S1 entity from SBERT
SBERT_MIN_SIM      = 0.25    # min cosine sim for SBERT candidates
TFIDF_MIN_SIM      = 0.08    # min cosine sim for TF-IDF candidates
RULE_MIN_TOKENS    = 2       # min shared tokens for rule-based blocking
VAL_SPLIT          = 0.15    # validation fraction
NEG_RATIO          = 5       # hard negatives per positive
XGB_THRESHOLD      = 0.5     # will be optimized later
BATCH_SIZE_SBERT   = 512     # reduce if OOM
BATCH_SIZE_CROSS   = 32      # reduce if OOM

print("✅ Imports and config done")

Device: cpu
Drive path exists: True
Contents: ['Documentation_template.md', '.DS_Store', 'README.md', 'dataset', 'utils']
✅ Imports and config done


In [ ]:
%%time
print("Loading training data...")

train_s1 = pd.read_csv(TRAIN_DIR / 'train_source1.tsv', sep='\t')
train_s2 = pd.read_csv(TRAIN_DIR / 'train_source2.tsv', sep='\t')
train_s3 = pd.read_csv(TRAIN_DIR / 'train_source3.tsv', sep='\t')
train_gt = pd.read_csv(TRAIN_DIR / 'train_ground_truth.tsv', sep='\t')

print(f"\n{'Dataset':<20} {'Rows':>12}")
print("=" * 35)
for name, df in [('Train S1', train_s1), ('Train S2', train_s2), ('Train S3', train_s3), ('Train GT', train_gt)]:
    print(f"{name:<20} {len(df):>12,}")

n_singletons = train_gt['matched_entity_ids'].isna().sum()
n_matched = train_gt['matched_entity_ids'].notna().sum()
print(f"\nSingletons (no match): {n_singletons:,} ({n_singletons/len(train_gt)*100:.1f}%)")
print(f"Has matches:          {n_matched:,} ({n_matched/len(train_gt)*100:.1f}%)")
print("✅ Train data loaded into memory (RAM usage: only ~3 GB)!")


Loading training data...

Dataset                      Rows
Train S1                2,206,821
Train S2                5,034,616
Train S3                5,285,603
Train GT                2,206,821

Singletons (no match): 123,247 (5.6%)
Has matches:          2,083,574 (94.4%)
✅ Train data loaded into memory (RAM usage: only ~3 GB)!
CPU times: user 20.1 s, sys: 1.35 s, total: 21.4 s
Wall time: 1min 4s


In [ ]:
# --- Country distribution ---
print("=== Train S1 country distribution ===")
print(train_s1['country'].value_counts())
print()

# --- Match count distribution ---
matched_gt = train_gt[train_gt['matched_entity_ids'].notna()].copy()
matched_gt['n_matches'] = matched_gt['matched_entity_ids'].apply(lambda x: len(str(x).split(',')))
print("=== Number of matches per S1 entity ===")
print(matched_gt['n_matches'].describe())
print()
print(matched_gt['n_matches'].value_counts().sort_index())
print()

# --- Sample matched pairs ---
print("=== Sample matched pairs (for intuition) ===")
sample_gt = matched_gt.head(5)
train_s23 = pd.concat([train_s2, train_s3]).set_index('entity_id')
train_s1_idx = train_s1.set_index('entity_id')

for _, row in sample_gt.iterrows():
    s1_id = row['source1_entity_id']
    s1_rec = train_s1_idx.loc[s1_id]
    matched_ids = row['matched_entity_ids'].split(',')
    print(f"\n--- {s1_id} ---")
    print(f"  S1 Name:    {s1_rec['business_name']}")
    print(f"  S1 Address: {s1_rec['business_address']}")
    print(f"  S1 Country: {s1_rec['country']}")
    for m_id in matched_ids[:3]:  # show first 3 matches
        m_id = m_id.strip()
        if m_id in train_s23.index:
            m_rec = train_s23.loc[m_id]
            print(f"  -> {m_id}: {m_rec['business_name']} | {m_rec['business_address']}")

# --- Null check ---
print("\n=== Nulls in train S1 ===")
print(train_s1.isnull().sum())
print("\n=== Nulls in train S2 ===")
print(train_s2.isnull().sum())
print("\n=== Nulls in train S3 ===")
print(train_s3.isnull().sum())

# Cleanup
del matched_gt, train_s23, train_s1_idx
gc.collect()

=== Train S1 country distribution ===
country
US       1323633
India     883188
Name: count, dtype: int64

=== Number of matches per S1 entity ===
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: n_matches, dtype: float64

n_matches
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

=== Sample matched pairs (for intuition) ===

--- S1-965667 ---
  S1 Name:    Maure Williams Colombier Inc
  S1 Address: 85 Wayne Avenue, Ticonderoga, NY
  S1 Country: US
  -> S2-681193310: Maure Wilblims Colombier Inc | nan
  -> S2-743505751: Maure Williams Colombier | nan
  -> S3-775321672: Dréxkor | 85 Wanye Avenue, Ticonderoga Townshiip, New York

--- S1-55344266 ---
  S1 Name:    Raj Investments LLP
  S1 Address: 6(29), C.I.T. Colony, 2Nd Main Road Mylapo

37

In [ ]:
# ============================================================
# LEGAL SUFFIXES & BUSINESS PREFIXES
# ============================================================
LEGAL_SUFFIXES = [
    # English (longest first)
    'private limited', 'pvt ltd', 'pvt limited', 'limited liability partnership',
    'limited liability company', 'incorporated', 'corporation',
    'limited', 'inc', 'corp', 'llc', 'llp', 'lp', 'plc', 'ltd', 'pvt', 'co',
    'private', 'public',
    # French
    'societe a responsabilite limitee', 'societe anonyme',
    'societe par actions simplifiee', 'societe civile immobiliere',
    'sarl', 'sas', 'sasu', 'eurl', 'sa', 'sci', 'gie', 'snc', 'scp',
    # Generic Business Qualifiers
    'group', 'holding', 'holdings', 'enterprises', 'enterprise',
    'industries', 'industry', 'services', 'solutions', 'technologies',
    'technology', 'international', 'intl',
    'praaivett limittedd', 'praivet limited', 'limittedd', 'praaivett', 'elelpii',
    'piraivet limitet', 'piraiveett limitett', 'limitet',
]

BUSINESS_PREFIXES = [
    'ms', 'the', 'ets', 'ste', 'cie', 'cabinet', 'groupe', 'dr', 'shri', 'messrs'
]

# Pre-compile regex patterns (sorted by length descending)
_SUFFIX_PATTERNS = [
    (re.compile(r'\b' + re.escape(s) + r'\b'), s)
    for s in sorted(LEGAL_SUFFIXES, key=len, reverse=True)
]

_PREFIX_PATTERNS = [
    (re.compile(r'^\b' + re.escape(p) + r'\b\s*'), p)
    for p in sorted(BUSINESS_PREFIXES, key=len, reverse=True)
]

ADDRESS_ABBR = {
    'rd': 'road', 'st': 'street', 'ave': 'avenue', 'blvd': 'boulevard',
    'dr': 'drive', 'ln': 'lane', 'ct': 'court', 'pl': 'place',
    'sq': 'square', 'hwy': 'highway', 'pkwy': 'parkway',
    'apt': 'apartment', 'ste': 'suite', 'fl': 'floor',
    'bldg': 'building', 'dept': 'department', 'mt': 'mount',
    'ft': 'fort', 'n': 'north', 's': 'south', 'e': 'east', 'w': 'west',
    'ne': 'northeast', 'nw': 'northwest', 'se': 'southeast', 'sw': 'southwest',
    # French
    'bd': 'boulevard', 'av': 'avenue', 'imp': 'impasse',
}


def normalize_text(text: str) -> str:
    """Core text normalization."""
    if pd.isna(text) or not text:
        return ''
    text = str(text)
    # Transliterate unicode to ASCII (handles accents, Devanagari→Latin)
    text = unidecode(text)
    text = text.lower().strip()
    # Standardize common symbols & abbreviations
    text = text.replace('&', ' and ')
    text = text.replace('@', ' at ')
    text = text.replace('#', ' number ')
    text = re.sub(r'\bm/s\b|\bm\s*s\b', 'ms', text)  # normalize M/S to ms
    # Remove all punctuation except spaces and alphanumerics
    text = re.sub(r'[^\w\s]', ' ', text)
    # Collapse whitespace
    return re.sub(r'\s+', ' ', text).strip()


def extract_name_components(name: str):
    """
    Decomposes business name into:
    1. core_name: primary distinctive brand (given HIGHEST weight)
    2. prefix: formal prefix (e.g. 'the', 'ms', 'dr')
    3. suffix: legal corporate form (e.g. 'pvt ltd', 'inc', 'sarl')
    """
    if not name:
        return '', '', ''

    # 1. Extract prefix from the start of the string
    extracted_prefix = ''
    for pat, p in _PREFIX_PATTERNS:
        if pat.search(name):
            extracted_prefix = p
            name = pat.sub('', name).strip()
            break

    # 2. Extract all legal suffixes / qualifiers (handles 'enterprises limited')
    extracted_suffixes = []
    for pat, s in _SUFFIX_PATTERNS:
        if pat.search(name):
            extracted_suffixes.append(s)
            name = pat.sub('', name).strip()

    core_name = re.sub(r'\s+', ' ', name).strip()
    suffix_str = ' '.join(extracted_suffixes)
    return core_name, extracted_prefix, suffix_str


def expand_address_abbr(address: str) -> str:
    """Expand address abbreviations."""
    tokens = address.split()
    return ' '.join(ADDRESS_ABBR.get(t, t) for t in tokens)


def extract_numbers(text: str) -> str:
    """Extract all numeric tokens from text."""
    return ' '.join(re.findall(r'\b\d+\b', text))


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply decomposed preprocessing to a source dataframe."""
    df = df.copy()
    df['business_name'] = df['business_name'].fillna('')
    df['business_address'] = df['business_address'].fillna('')
    df['country'] = df['country'].fillna('')

    # 1. Full normalized name
    df['clean_name'] = df['business_name'].apply(normalize_text)

    # 2. Decompose into Core Name, Prefix, and Suffix
    components = [extract_name_components(n) for n in df['clean_name']]
    df['clean_name_core'] = [c[0] for c in components]
    df['name_prefix']     = [c[1] for c in components]
    df['legal_suffix']    = [c[2] for c in components]

    # Keep legacy name for backward compatibility with other cells
    df['clean_name_nosuffix'] = df['clean_name_core']

    # 3. Clean address and extract address numbers
    df['clean_address'] = df['business_address'].apply(normalize_text).apply(expand_address_abbr)
    df['addr_numbers'] = df['clean_address'].apply(extract_numbers)

    # 4. Combined text for embedding / TF-IDF (core name + address)
    df['combined_text'] = df['clean_name_core'] + ' | ' + df['clean_address']

    # 5. Core tokens for blocking (clean, distinctive tokens only)
    df['name_tokens'] = df['clean_name_core'].apply(lambda x: set(x.split()) if x else set())

    return df


print("✅ Preprocessing functions defined")

# Test decomposition
test_names = [
    "The Prabhav Business Center Private Limited",
    "B+ Retail Inc",
    "M/S Apex Technologies",
    "Société Anonyme de Construction SARL",
    "Apex Enterprises Limited",
    "राम मार्केटिंग प्राइवेट लिमिटेड",
]

print(f"\n{'Original Name':<45} | {'Core Name':<25} | {'Prefix':<6} | {'Suffix'}")
print("=" * 100)
for n in test_names:
    norm = normalize_text(n)
    core, pre, suf = extract_name_components(norm)
    print(f"{n:<45} | {core:<25} | {pre:<6} | {suf}")


✅ Preprocessing functions defined

Original Name                                 | Core Name                 | Prefix | Suffix
The Prabhav Business Center Private Limited   | prabhav business center   | the    | private limited
B+ Retail Inc                                 | b retail                  |        | inc
M/S Apex Technologies                         | apex                      | ms     | technologies
Société Anonyme de Construction SARL          | de construction           |        | societe anonyme sarl
Apex Enterprises Limited                      | apex                      |        | enterprises limited
राम मार्केटिंग प्राइवेट लिमिटेड               | raam maarketting          |        | praaivett limittedd


In [ ]:
%%time
import gc

# 1. Preprocess Train S1
print("1/3 Preprocessing train_s1 (2.2M rows)...")
train_s1 = preprocess_dataframe(train_s1)
train_s1['country'] = train_s1['country'].astype('category')
gc.collect()

# 2. Preprocess Train S2
print("2/3 Preprocessing train_s2 (5.0M rows)...")
train_s2 = preprocess_dataframe(train_s2)
train_s2['country'] = train_s2['country'].astype('category')
gc.collect()

# 3. Preprocess Train S3
print("3/3 Preprocessing train_s3 (5.3M rows)...")
train_s3 = preprocess_dataframe(train_s3)
train_s3['country'] = train_s3['country'].astype('category')
gc.collect()

# 4. Combine S2 + S3 and immediately DELETE the separate copies to free RAM
print("\nCombining S2 and S3...")
train_s23 = pd.concat([train_s2, train_s3], ignore_index=True)
del train_s2, train_s3  # Frees ~10 million rows from memory!
gc.collect()

print(f"Train S1 rows:     {len(train_s1):,}")
print(f"Train S2+S3 rows:  {len(train_s23):,}")

# 5. Build zero-copy lookups (sets index in-place without duplicating the dataframes!)
print("\nIndexing lookups (zero-copy)...")
train_s1.set_index('entity_id', drop=False, inplace=True)
train_s23.set_index('entity_id', drop=False, inplace=True)

train_s1_lookup  = train_s1   # Zero extra memory!
train_s23_lookup = train_s23  # Zero extra memory!

gc.collect()
print("\n✅ All training data preprocessed successfully without crashing!")
print("ℹ️ Note: Test data will be preprocessed on-the-fly at Cell 24 for final predictions.")


1/3 Preprocessing train_s1 (2.2M rows)...
2/3 Preprocessing train_s2 (5.0M rows)...
3/3 Preprocessing train_s3 (5.3M rows)...

Combining S2 and S3...
Train S1 rows:     2,206,821
Train S2+S3 rows:  10,320,219

Indexing lookups (zero-copy)...

✅ All training data preprocessed successfully without crashing!
ℹ️ Note: Test data will be preprocessed on-the-fly at Cell 24 for final predictions.
CPU times: user 9min 6s, sys: 3.71 s, total: 9min 10s
Wall time: 9min 10s


In [ ]:
import os

# Create a project folder in your Drive
SAVE_DIR = '/content/drive/MyDrive/amazon/amazon_ml_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)

print("💾 Saving preprocessed data to Google Drive...")
train_s1.to_parquet(os.path.join(SAVE_DIR, 'train_s1.parquet'))
train_s23.to_parquet(os.path.join(SAVE_DIR, 'train_s23.parquet'))

print("✅ Saved to Google Drive! Even if Colab disconnects, your data is 100% safe.")


💾 Saving preprocessed data to Google Drive...
✅ Saved to Google Drive! Even if Colab disconnects, your data is 100% safe.


In [ ]:
import os
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/amazon/amazon_ml_challenge/'

print("⚡ Loading preprocessed data from Google Drive in 3 seconds...")
train_s1 = pd.read_parquet(os.path.join(SAVE_DIR, 'train_s1.parquet'))
train_s23 = pd.read_parquet(os.path.join(SAVE_DIR, 'train_s23.parquet'))

# Restore index lookups
train_s1.set_index('entity_id', drop=False, inplace=True)
train_s23.set_index('entity_id', drop=False, inplace=True)
train_s1_lookup = train_s1
train_s23_lookup = train_s23

print(f"✅ Loaded {len(train_s1):,} S1 rows and {len(train_s23):,} S2+S3 rows! Ready to run Cell 9.")


⚡ Loading preprocessed data from Google Drive in 3 seconds...


In [ ]:
# Parse ground truth into a dict: {s1_id: set(matched_ids)}
def parse_ground_truth(gt_df):
    gt_dict = {}
    for _, row in gt_df.iterrows():
        s1_id = row['source1_entity_id']
        if pd.isna(row['matched_entity_ids']) or not str(row['matched_entity_ids']).strip():
            gt_dict[s1_id] = set()
        else:
            gt_dict[s1_id] = set(m.strip() for m in str(row['matched_entity_ids']).split(',') if m.strip())
    return gt_dict

full_gt = parse_ground_truth(train_gt)
print(f"Total S1 entities in ground truth: {len(full_gt):,}")
print(f"Singletons: {sum(1 for v in full_gt.values() if len(v)==0):,}")

# Split by S1 entity ID — no leakage
all_s1_ids = list(full_gt.keys())
np.random.seed(42)
np.random.shuffle(all_s1_ids)
split_idx = int(len(all_s1_ids) * (1 - VAL_SPLIT))

train_s1_ids = set(all_s1_ids[:split_idx])
val_s1_ids   = set(all_s1_ids[split_idx:])

train_gt_dict = {k: v for k, v in full_gt.items() if k in train_s1_ids}
val_gt_dict   = {k: v for k, v in full_gt.items() if k in val_s1_ids}

print(f"\nTrain S1 entities: {len(train_gt_dict):,}")
print(f"Val   S1 entities: {len(val_gt_dict):,}")
print(f"Val singletons:    {sum(1 for v in val_gt_dict.values() if len(v)==0):,}")

In [ ]:
def compute_f05_macro(predictions: dict, ground_truth: dict) -> dict:
    """
    Compute macro-averaged F0.5 across all S1 entities.
    predictions:   {s1_id: set(predicted_match_ids)}
    ground_truth:  {s1_id: set(true_match_ids)}
    Returns dict with f05, precision, recall, and per-entity scores.
    """
    f05_scores = []
    precisions = []
    recalls = []

    for s1_id in ground_truth:
        true = ground_truth[s1_id]
        pred = predictions.get(s1_id, set())
        if isinstance(pred, list):
            pred = set(pred)
        if isinstance(true, list):
            true = set(true)

        if len(pred) == 0 and len(true) == 0:
            f05_scores.append(1.0)
            precisions.append(1.0)
            recalls.append(1.0)
            continue
        if len(pred) == 0:
            f05_scores.append(0.0)
            precisions.append(0.0)
            recalls.append(0.0)
            continue
        if len(true) == 0:
            f05_scores.append(0.0)
            precisions.append(0.0)
            recalls.append(1.0)
            continue

        tp = len(pred & true)
        fp = len(pred - true)
        fn = len(true - pred)

        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0

        if p + r > 0:
            f05 = (1.25 * p * r) / (0.25 * p + r)
        else:
            f05 = 0.0

        f05_scores.append(f05)
        precisions.append(p)
        recalls.append(r)

    return {
        'f05': np.mean(f05_scores),
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'n_entities': len(f05_scores),
    }

print("✅ F0.5 evaluator ready")

# Quick test
test_pred = {'A': {'X', 'Y'}, 'B': set()}
test_true = {'A': {'X', 'Z'}, 'B': set()}
print("Test:", compute_f05_macro(test_pred, test_true))

In [ ]:
%%time
import gc
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

def tfidf_blocking(s1_df, s23_df, top_k=25, min_sim=0.08, batch_size=100):
    """
    Ultra-lean, pure sparse TF-IDF blocking for Kaggle.
    - NEVER calls .toarray() (eliminates 5 GB dense memory spikes!)
    - Fits vocabulary on a sample in 5 seconds
    - Uses only ~15 MB per batch instead of 5,000 MB
    """
    candidates = {eid: set() for eid in s1_df['entity_id']}
    countries = s1_df['country'].dropna().unique()

    for c in countries:
        s1_c = s1_df[s1_df['country'] == c]
        s23_c = s23_df[s23_df['country'] == c]
        if len(s1_c) == 0 or len(s23_c) == 0:
            continue

        print(f"\n  [Country: {c}] S1: {len(s1_c):,} | S2+S3: {len(s23_c):,}")

        # 1. Compact 3-4 char n-gram vectorizer (cuts matrix memory in half)
        vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 4),
            min_df=5,
            max_features=75000,
            sublinear_tf=True,
            dtype=np.float32
        )

        # 2. Fit vocabulary on a sample in 5 seconds (avoids 10 GB memory spike)
        sample_size = min(300000, len(s23_c))
        print(f"    Fitting vocabulary on {sample_size:,} sample texts...")
        sample_texts = s23_c['combined_text'].sample(sample_size, random_state=42)
        vectorizer.fit(sample_texts)
        del sample_texts
        gc.collect()

        # 3. Transform into compact sparse CSR matrices
        print(f"    Transforming matrices...")
        s23_matrix = vectorizer.transform(s23_c['combined_text'])
        s1_matrix  = vectorizer.transform(s1_c['combined_text'])

        s1_ids  = s1_c['entity_id'].values
        s23_ids = s23_c['entity_id'].values

        # 4. Pure Sparse Dot Product (Never converts to dense!)
        for start in tqdm(range(0, len(s1_ids), batch_size), desc=f"    Sparse TF-IDF ({c})"):
            end = min(start + batch_size, len(s1_ids))
            batch_mat = s1_matrix[start:end]

            # Sparse matrix product: shape (batch_size, n_s23) -> stays SPARSE!
            sim_sparse = batch_mat @ s23_matrix.T

            for j in range(sim_sparse.shape[0]):
                row = sim_sparse.getrow(j)
                data = row.data
                indices = row.indices

                if len(data) == 0:
                    continue

                mask = data >= min_sim
                v_data = data[mask]
                v_idx  = indices[mask]

                if len(v_data) > top_k:
                    top_sub = np.argpartition(v_data, -top_k)[-top_k:]
                    selected_ids = v_idx[top_sub]
                else:
                    selected_ids = v_idx

                s1_id = s1_ids[start + j]
                candidates[s1_id] = set(s23_ids[selected_ids])

            del sim_sparse

        del s23_matrix, s1_matrix, vectorizer
        gc.collect()

    total_cands = sum(len(v) for v in candidates.values())
    avg_cands = total_cands / max(1, len(candidates))
    print(f"\n  ✅ TF-IDF blocking complete: {total_cands:,} total candidates (avg {avg_cands:.1f}/entity)")
    return candidates

# Validation sample of 5,000 for quick recall evaluation
val_s1_sample = train_s1[train_s1['entity_id'].isin(val_s1_ids)].sample(min(5000, len(val_s1_ids)), random_state=42)
tfidf_cands_val = tfidf_blocking(val_s1_sample, train_s23, top_k=TFIDF_TOP_K, min_sim=TFIDF_MIN_SIM, batch_size=100)


Blocking 10,000 validation S1 entities against 10,320,219 S2+S3 entities...

  [Country: US] S1: 5,995 entities | S2+S3: 6,186,873 entities
    Fitting TF-IDF vectorizer for US...


    TF-IDF Matching (US):   0%|          | 0/30 [00:00<?, ?it/s]


  [Country: India] S1: 4,005 entities | S2+S3: 4,133,346 entities
    Fitting TF-IDF vectorizer for India...


    TF-IDF Matching (India):   0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
import os
import pickle

SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)

print("💾 Saving Cell 9 results to Google Drive...")

# 1. Save the candidates dictionary
with open(os.path.join(SAVE_DIR, 'tfidf_cands_val.pkl'), 'wb') as f:
    pickle.dump(tfidf_cands_val, f, protocol=pickle.HIGHEST_PROTOCOL)

# 2. Save the validation dataframe used
val_s1_df.to_parquet(os.path.join(SAVE_DIR, 'val_s1_df.parquet'))

print("✅ Saved! 'tfidf_cands_val.pkl' and 'val_s1_df.parquet' are permanently backed up.")


In [ ]:
import os
import pickle
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'

print("⚡ Loading Cell 9 results from Google Drive...")

# Load candidates dictionary
with open(os.path.join(SAVE_DIR, 'tfidf_cands_val.pkl'), 'rb') as f:
    tfidf_cands_val = pickle.load(f)

# Load validation dataframe
val_s1_df = pd.read_parquet(os.path.join(SAVE_DIR, 'val_s1_df.parquet'))

print(f"✅ Loaded {len(tfidf_cands_val):,} TF-IDF validation candidates in 1 second! Ready for Cell 11.")


In [ ]:
%%time

def rule_based_blocking(s1_df, s23_df, min_shared_tokens=2):
    """
    Rule-based blocking: shared name tokens + same country.
    Returns {s1_entity_id: set(candidate_entity_ids)}.
    """
    # Build inverted index: token → list of (entity_id, country)
    print("  Building inverted index...")
    token_index = defaultdict(list)
    s23_countries = dict(zip(s23_df['entity_id'], s23_df['country']))

    for _, row in tqdm(s23_df.iterrows(), total=len(s23_df), desc="  Indexing S2+S3"):
        eid = row['entity_id']
        for token in row['name_tokens']:
            if len(token) > 2:  # skip very short tokens
                token_index[token].append(eid)

    print(f"  Inverted index: {len(token_index):,} unique tokens")

    # For each S1 entity, find candidates with ≥ min_shared_tokens in common
    candidates = {}
    for _, row in tqdm(s1_df.iterrows(), total=len(s1_df), desc="  Rule blocking"):
        s1_id = row['entity_id']
        s1_country = row['country']
        s1_tokens = row['name_tokens']

        cand_counts = defaultdict(int)
        for token in s1_tokens:
            if len(token) > 2:
                for cand_id in token_index.get(token, []):
                    cand_counts[cand_id] += 1

        cands = set()
        for cand_id, count in cand_counts.items():
            if count >= min_shared_tokens and s23_countries.get(cand_id) == s1_country:
                cands.add(cand_id)
        candidates[s1_id] = cands

    total_cands = sum(len(v) for v in candidates.values())
    avg_cands = total_cands / len(candidates) if candidates else 0
    print(f"  Rule-based blocking: {total_cands:,} total candidates, avg {avg_cands:.1f}/entity")
    return candidates


rule_cands_val = rule_based_blocking(
    val_s1_df, train_s23, min_shared_tokens=RULE_MIN_TOKENS
)